# EDA: Students Performance Dataset

## Цель анализа
В этом ноутбуке я провожу первичный анализ датасета **StudentsPerformance**:
- изучаю структуру данных;
- проверяю распределения баллов;
- сравниваю результаты между группами;
- ищу факторы, которые могут быть связаны с успеваемостью.

## Почему этот проект полезен
Это пример базового exploratory data analysis:
от чтения данных и очистки названий столбцов до агрегаций, фильтрации и формулировки выводов.

In [1]:
import pandas as pd
import numpy as np

students = pd.read_csv(
    "https://stepik.org/media/attachments/course/4852/StudentsPerformance.csv"
)

students = students.rename(columns={
    "parental level of education": "parental_level_of_education",
    "test preparation course": "test_preparation_course",
    "math score": "math_score",
    "reading score": "reading_score",
    "writing score": "writing_score",
    "race/ethnicity": "race_ethnicity"
})

students.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## 1. Первичный обзор данных
Сначала проверяю размер таблицы, типы столбцов и базовые статистики.

In [2]:
students.shape, students.dtypes

((1000, 8),
 gender                         object
 race_ethnicity                 object
 parental_level_of_education    object
 lunch                          object
 test_preparation_course        object
 math_score                      int64
 reading_score                   int64
 writing_score                   int64
 dtype: object)

In [3]:
students.describe(include="all")

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
count,1000,1000,1000,1000,1000,1000.00000,1000.000000,1000.000000
unique,2,5,6,2,2,NaN,NaN,NaN
top,female,group C,some college,standard,none,NaN,NaN,NaN
freq,518,319,226,645,642,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,66.08900,69.169000,68.054000
std,NaN,NaN,NaN,NaN,NaN,15.16308,14.600192,15.195657
min,NaN,NaN,NaN,NaN,NaN,0.00000,17.000000,10.000000
25%,NaN,NaN,NaN,NaN,NaN,57.00000,59.000000,57.750000
50%,NaN,NaN,NaN,NaN,NaN,66.00000,70.000000,69.000000
75%,NaN,NaN,NaN,NaN,NaN,77.00000,79.000000,79.000000


## 2. Базовые срезы по успеваемости
Сравню средние баллы по полу и по нескольким категориальным признакам.

In [4]:
students.groupby("gender")[["math_score", "reading_score", "writing_score"]].mean().round(2)

,math_score,reading_score,writing_score
gender,,,
female,63.63,72.61,72.47
male,68.73,65.47,63.31


In [5]:
students.groupby("race_ethnicity")[["math_score", "reading_score", "writing_score"]].mean().round(2)

,math_score,reading_score,writing_score
race_ethnicity,,,
group A,61.63,64.67,62.67
group B,63.45,67.35,65.60
group C,64.46,69.10,67.83
group D,67.36,70.03,70.15
group E,73.82,73.03,71.41


In [6]:
students.groupby("test_preparation_course")[["math_score", "reading_score", "writing_score"]].mean().round(2)

,math_score,reading_score,writing_score
test_preparation_course,,,
completed,69.70,73.89,74.42
none,64.08,66.53,64.50


## 3. Влияние типа питания
Один из интересных признаков — `lunch`. Посмотрю, отличаются ли средние результаты у групп.

In [7]:
students_free_reduced = students[students["lunch"] == "free/reduced"]
students_standard = students[students["lunch"] == "standard"]

pd.DataFrame({
    "free/reduced_mean": students_free_reduced[["math_score", "reading_score", "writing_score"]].mean(),
    "standard_mean": students_standard[["math_score", "reading_score", "writing_score"]].mean(),
    "free/reduced_var": students_free_reduced[["math_score", "reading_score", "writing_score"]].var(),
    "standard_var": students_standard[["math_score", "reading_score", "writing_score"]].var(),
}).round(2)

,free/reduced_mean,standard_mean,free/reduced_var,standard_var
math_score,58.92,70.03,229.82,186.42
reading_score,64.65,71.65,221.87,191.29
writing_score,63.02,70.82,238.20,205.62


## 4. Выборки и фильтрация
Ниже несколько типичных операций EDA: фильтрация, поиск лучших наблюдений и выбор нужных столбцов.

In [9]:
high_writing = students.query("writing_score > writing_score.mean()")
high_writing[["gender", "parental_level_of_education", "writing_score"]].head()

,gender,parental_level_of_education,writing_score
0,female,bachelor's degree,74
1,female,some college,88
2,female,master's degree,93
4,male,some college,75
5,female,associate's degree,78


In [10]:
score_columns = [col for col in students.columns if "score" in col]
students[score_columns].head()

,math_score,reading_score,writing_score
0,72,72,74
1,69,90,88
2,90,95,93
3,47,57,44
4,76,78,75


In [11]:
top_math_by_gender = (
    students.sort_values(["gender", "math_score"], ascending=False)
            .groupby("gender")
            .head(5)
            [["gender", "race_ethnicity", "math_score", "reading_score", "writing_score"]]
)

top_math_by_gender

,gender,race_ethnicity,math_score,reading_score,writing_score
149,male,group E,100,100,93
623,male,group A,100,96,86
625,male,group D,100,97,99
916,male,group E,100,100,100
306,male,group E,99,87,81
451,female,group E,100,92,97
458,female,group E,100,100,100
962,female,group E,100,100,100
114,female,group E,99,100,100
263,female,group E,99,93,90


## 5. Группировки по нескольким признакам
Здесь смотрю, как меняется средний балл по математике сразу по двум категориям.

In [12]:
mean_math_by_group = (
    students.groupby(["gender", "race_ethnicity"], as_index=False)
            .agg(mean_math_score=("math_score", "mean"))
            .sort_values("mean_math_score", ascending=False)
)

mean_math_by_group.head(10)

,gender,race_ethnicity,mean_math_score
9,male,group E,76.746479
4,female,group E,70.811594
8,male,group D,69.413534
7,male,group C,67.611511
6,male,group B,65.930233
3,female,group D,65.248062
5,male,group A,63.735849
2,female,group C,62.033333
1,female,group B,61.403846
0,female,group A,58.527778


## Выводы

### Что удалось увидеть
1. Разные предметы показывают разные паттерны по группам: обычно мальчики чуть сильнее в математике, а девочки — в чтении и письме.
2. Прохождение preparation course связано с более высокими результатами почти по всем предметам.
3. Группа `standard lunch` в среднем показывает более высокие баллы, чем `free/reduced`, что может указывать на социально-экономические различия.
4. Средние значения удобно дополнять разбором дисперсий: это помогает увидеть не только уровень результатов, но и их разброс.
5. Даже на небольшом датасете простые операции `groupby`, `query`, сортировка и фильтрация уже дают полезные аналитические наблюдения.